In [ ]:
# Imports

import os
import glob
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage

import gradio as gr

load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}...")
else:
    print("OpenRouter API Key not set")



In [ ]:
# Constants

MODEL    = "openai/gpt-4.1-mini"
DB_NAME  = "vectordb"
KB_PATH  = "knowledge_base"




## Step 1 — Load the knowledge base

The documentation knowledge base is organized by source so that the RAG system can retrieve information from multiple developer documentation sites.

• **langchain/** — contains documentation pages from the LangChain framework, including concepts such as agents, retrievers, chains, vector stores, and prompt templates.

• **huggingface/** — contains documentation from the Hugging Face ecosystem, covering topics like transformers, pipelines, datasets, and model usage.

• **fastapi/** — contains documentation for the FastAPI framework, including routing, dependency injection, request handling, and API development patterns.

• **python_docs/** — contains selected sections of the official Python documentation that explain core language features and standard library modules.

All documentation pages are stored as Markdown files generated by the crawler. These files form the knowledge base that will later be processed during the ingestion stage.



In [ ]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

# project root
PROJECT_ROOT = Path.cwd()
# knowledge base directory
DATA_PATH = "knowledge_base"


def load_documents():
    """
    Load all markdown documentation from the knowledge base
    """
    loader = DirectoryLoader(
        DATA_PATH,
        glob="**/*.md",
        loader_cls=lambda path: TextLoader(path, encoding="utf-8"),
        show_progress=True,
        silent_errors=True
    )
    documents = loader.load()
    print(f"Loaded {len(documents)} markdown files")
    return documents

#chunking the documents based on markdown headers


def split_documents(documents):
    """
    Split documents based on markdown headers
    """
    headers = [
        ("#", "header1"),
        ("##", "header2"),
        ("###", "header3"),
    ]

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers
    )

    chunks = []

    for doc in documents:

        splits = splitter.split_text(doc.page_content)

        for split in splits:

            split.metadata = doc.metadata

            chunks.append(split)

    print(f"Created {len(chunks)} chunks")

    return chunks


if __name__ == "__main__":

    docs = load_documents()

    chunks = split_documents(docs)

    print("Example chunk:\n")
    print(chunks[0].page_content[:500])

In [ ]:
# Create embeddings and vector store
import shutil

print("Loading HuggingFace embedding model (all-MiniLM-L6-v2)...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# Delete existing collection so we always rebuild from current knowledge base
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
    print(f"Deleted existing vector store at '{DB_NAME}'")

print("Building ChromaDB vector store...")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

# Persist database if method exists (for backward compatibility)
if hasattr(vectorstore, "persist"):
    vectorstore.persist()

count = getattr(vectorstore, "_collection", getattr(vectorstore, "collection", None)).count() if hasattr(vectorstore, "_collection") or hasattr(vectorstore, "collection") else 0

print("\nVector store ready:")
print(f"Vectors stored: {count:,}")
print(f"Embedding model: all-MiniLM-L6-v2")

In [ ]:


# Set up retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)



llm = ChatOpenAI(
    model=MODEL,  
    temperature=0,
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

print("Retriever: top-8 documentation chunks by cosine similarity")
print(f"LLM via OpenRouter: {MODEL} (temperature=0 for factual answers)")

In [ ]:
# Test the retriever with a developer documentation query

import os

test_query = "How do LangChain retrievers work?"

retrieved_docs = retriever.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} documentation chunks:\n")

for i, doc in enumerate(retrieved_docs, 1):

    source = doc.metadata.get("source", "unknown")
    site   = doc.metadata.get("doc_type", "docs")

    print(f"[{i}] {site} / {os.path.basename(source)}")
    print(f"    {doc.page_content[:150]}...\n")

In [ ]:
SYSTEM_PROMPT = """\
You are a developer documentation assistant.

You have access to excerpts from developer documentation sources such as
LangChain, HuggingFace Transformers, FastAPI, and Python documentation.

When answering:
- Use the provided context excerpts to produce accurate, concise explanations
- Prefer information directly supported by the retrieved documentation
- Mention the documentation source when relevant (for example: LangChain docs, HuggingFace docs, FastAPI docs)
- If the context does not contain enough information, clearly state that the documentation does not provide the answer
- Do not invent APIs, functions, or configuration options that are not present in the provided documentation
- Format responses using Markdown for clarity (bullet points, code blocks, or tables when useful)

The goal is to help developers understand how to use frameworks, APIs, and programming concepts based on the documentation.

Context:
{context}
"""


def answer_question(question: str, history) -> str:
    """Retrieve relevant documentation chunks and generate an answer."""

    docs = retriever.invoke(question)[:4]

    context = "\n\n---\n\n".join(
        f"[Source: {d.metadata.get('doc_type','docs')} / {os.path.basename(d.metadata.get('source','unknown'))}]\n{d.page_content}"
        for d in docs
    )

    system_prompt = SYSTEM_PROMPT.format(context=context)

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ])

    return response.content


# Quick test
print(answer_question("How do LangChain retrievers work?", []))

 Gradio Chat UI

In [ ]:
EXAMPLE_QUESTIONS = [
    "How do LangChain retrievers work?",
    "What is a HuggingFace pipeline and how do I use it?",
    "How do FastAPI dependencies work?",
    "What is async programming in Python?",
    "How do I create a vector store using LangChain?",
    "What is the difference between transformers pipeline and AutoModel?",
    "How do I build an API using FastAPI?",
    "How does RetrievalQA work in LangChain?",
]

with gr.Blocks(title="Developer Documentation RAG Assistant", theme=gr.themes.Soft()) as ui:

    gr.Markdown("""
    # Developer Documentation RAG Assistant

    Ask questions about developer frameworks and programming concepts.
    Answers are generated using a Retrieval-Augmented Generation (RAG) pipeline
    grounded in official developer documentation.

    **Knowledge base includes documentation from:**
    - LangChain
    - HuggingFace Transformers
    - FastAPI
    - Python documentation

    The assistant retrieves relevant documentation chunks from the knowledge base
    and uses them as context for generating answers.

    > Responses are based on the retrieved documentation context.
    > If the documentation does not contain the answer, the assistant will say so.
    """)

    gr.Markdown(
    "Built with LangChain, ChromaDB, HuggingFace embeddings, and OpenRouter."
     )

    chatbot = gr.ChatInterface(
        fn=answer_question,
        type="messages",
        examples=EXAMPLE_QUESTIONS,
        chatbot=gr.Chatbot(
            height=520,
            show_label=False,
            render_markdown=True,
            type="messages"
        ),
        textbox=gr.Textbox(
            placeholder="Ask a developer documentation question...",
            show_label=False
        ),
    )

print("UI built.")

In [ ]:
ui.launch()